# Air Quality Index (AQI) - Exploratory Data Analysis (EDA)

This notebook explores the historical air pollution dynamics, seasonal patterns, missing value distributions, and cross-pollutant correlations for Indian cities (specifically Delhi).

### Key Questions:
1. What is the extent of missingness across key criteria pollutants ($PM_{2.5}, PM_{10}, NO_2, SO_2, CO, O_3$)?
2. What are the macro temporal trends and annual seasonal cycles?
3. How strongly do specific particulate matter and gaseous pollutants correlate with the overall AQI?
4. What is the historical distribution across the official Central Pollution Control Board (CPCB) health categories?


In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add src to python path
repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))
if str(repo_root / "src") not in sys.path:
    sys.path.append(str(repo_root / "src"))

from src.config import Config, get_aqi_category
from src.data_loader import load_raw_data
from src.preprocessing import clean_and_prepare_timeseries

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({"font.size": 11, "figure.autolayout": True})


## 1. Data Ingestion
Load raw observations for Delhi (or fallback to calibrated synthetic series if raw Kaggle CSV is not downloaded yet).

In [ ]:
config = Config(city="Delhi")
raw_df = load_raw_data(city=config.city)
print(f"Dataset shape: {raw_df.shape}")
print(raw_df.head())


## 2. Missing Value Analysis
Assess missing observation percentages per pollutant column.

In [ ]:
cols = ["PM2.5", "PM10", "NO2", "SO2", "CO", "O3", "AQI"]
missing_counts = raw_df[cols].isna().sum()
missing_pct = (missing_counts / len(raw_df)) * 100.0

missing_df = pd.DataFrame({"Missing Count": missing_counts, "Missing Percentage (%)": missing_pct.round(2)})
print(missing_df)

fig, ax = plt.subplots(figsize=(8, 4))
missing_pct.plot(kind="bar", color="#e74c3c", ax=ax, edgecolor="black", alpha=0.85)
ax.set_title("Missing Values Percentage by Pollutant", fontsize=13, fontweight="bold")
ax.set_ylabel("Missing Percentage (%)")
ax.set_ylim(0, max(25, missing_pct.max() + 5))
for i, v in enumerate(missing_pct):
    ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.show()


## 3. Continuous Time Series Preparation
Apply time-based interpolation and cyclical calendar features.

In [ ]:
clean_df = clean_and_prepare_timeseries(raw_df, pollutant_cols=config.pollutant_cols, target_col="AQI")
print(f"Cleaned time series length: {len(clean_df)} continuous days")
print(clean_df.describe().round(2))


## 4. Multi-Year Trends & Winter Smog Inversion
Visualizing PM2.5, PM10, and AQI over the 2015-2020 timeline. Notice the recurring winter spikes (Nov-Jan) and monsoon washouts (Jul-Sep).

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(clean_df.index, clean_df["PM2.5"], color="#d35400", linewidth=1.2)
axes[0].set_ylabel("PM2.5 (ug/m3)")
axes[0].set_title("Particulate Matter (PM2.5) Timeline", fontsize=12, fontweight="bold")

axes[1].plot(clean_df.index, clean_df["PM10"], color="#2980b9", linewidth=1.2)
axes[1].set_ylabel("PM10 (ug/m3)")
axes[1].set_title("Particulate Matter (PM10) Timeline", fontsize=12, fontweight="bold")

axes[2].plot(clean_df.index, clean_df["AQI"], color="#c0392b", linewidth=1.3)
axes[2].axhline(200, color="orange", linestyle="--", alpha=0.7, label="Poor (200)")
axes[2].axhline(300, color="red", linestyle="--", alpha=0.7, label="Very Poor (300)")
axes[2].axhline(400, color="darkred", linestyle="--", alpha=0.7, label="Severe (400)")
axes[2].set_ylabel("Air Quality Index (AQI)")
axes[2].set_title("Air Quality Index (AQI) Timeline", fontsize=12, fontweight="bold")
axes[2].legend(loc="upper right")

plt.tight_layout()
plt.show()


## 5. Monthly Seasonality Analysis
Average monthly pollutant concentrations illustrating northern India's severe atmospheric inversion dynamics.

In [ ]:
monthly_avg = clean_df.groupby(clean_df.index.month)[cols].mean()
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(month_names, monthly_avg["AQI"], marker="o", color="#c0392b", linewidth=2.5, label="AQI")
ax1.plot(month_names, monthly_avg["PM2.5"], marker="s", color="#d35400", linewidth=2, label="PM2.5")
ax1.plot(month_names, monthly_avg["PM10"], marker="^", color="#2980b9", linewidth=2, label="PM10")
ax1.set_ylabel("Concentration / Index", fontsize=11)
ax1.set_title("Monthly Mean Variations in Pollutants and AQI", fontsize=13, fontweight="bold")
ax1.legend(loc="upper right")
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Cross-Pollutant Correlation Heatmap
Inter-pollutant Pearson correlation coefficients.

In [ ]:
corr = clean_df[cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols)))
ax.set_yticks(range(len(cols)))
ax.set_xticklabels(cols, rotation=45, ha="right", fontsize=10)
ax.set_yticklabels(cols, fontsize=10)

# Annotate correlation numbers
for i in range(len(cols)):
    for j in range(len(cols)):
        val = corr.iloc[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color="black" if abs(val) < 0.6 else "white", fontsize=9, fontweight="bold")

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Pearson Correlation Matrix Across Pollutants", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. CPCB AQI Category Distribution
Categorization under Indian CPCB thresholds.

In [ ]:
categories = [get_aqi_category(val)[0] for val in clean_df["AQI"]]
cat_series = pd.Series(categories).value_counts()
cat_order = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
cat_series = cat_series.reindex(cat_order).fillna(0)

colors = ["#2ecc71", "#a8e6cf", "#f1c40f", "#e67e22", "#e74c3c", "#7b1113"]

fig, ax = plt.subplots(figsize=(8, 4.5))
cat_series.plot(kind="bar", color=colors, ax=ax, edgecolor="black", alpha=0.9)
ax.set_title("Historical AQI Distribution by CPCB Category", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of Days")
for i, v in enumerate(cat_series):
    ax.text(i, v + 10, f"{int(v)}", ha="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.show()


## 8. Summary of Findings for GRU Modeling
1. **High Seasonality & Autocorrelation**: Air quality in Delhi exhibits strong periodicity, making cyclical sin/cos temporal representations indispensable.
2. **Dominant Particulate Drivers**: $PM_{2.5}$ and $PM_{10}$ correlate heavily ($r > 0.85$) with overall AQI, serving as critical leading indicators.
3. **Sequential Memory Requirements**: Weather inversions and wind stagnation create multi-day persistent smog episodes, which a 2-layer GRU with 14-day lookback is well suited to capture.
